In [ ]:
# ── WIDGETS ──────────────────────────────────────────────────
dbutils.widgets.text("catalog_param", "my_assessment")
catalog = dbutils.widgets.get("catalog_param")

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────
from pyspark.sql.functions import broadcast, col, count

In [ ]:
# ── READ TABLES ───────────────────────────────────────────────
orders_df      = spark.table(f"{catalog}.silver.orders_cleaned")
customers_df   = spark.table(f"{catalog}.silver.customers_cleaned")
products_df    = spark.table(f"{catalog}.silver.products_cleaned")
order_items_df = spark.table(f"{catalog}.silver.order_items_cleaned")

In [ ]:
# ── 1. BROADCAST JOIN ─────────────────────────────────────────
broadcast_join = orders_df \
    .join(broadcast(customers_df), "customer_id") \
    .join(order_items_df, "order_id") \
    .join(broadcast(products_df), "product_id")

print("✅ Broadcast join done")
broadcast_join.display()

In [ ]:
# ── 2. CACHING ────────────────────────────────────────────────
# Cache tables that are used multiple times
# Avoids re-reading from disk every time

#orders_df.cache()
#customers_df.cache()

# Trigger the cache by running an action
print(f"orders count: {orders_df.count()}")
print(f"customers count: {customers_df.count()}")
print("✅ Caching done")

In [ ]:
# ── 3. REPARTITION vs COALESCE ────────────────────────────────

# REPARTITION → increases or reshuffles partitions (full shuffle)
# Use when: you need MORE partitions or even distribution
orders_repartitioned = orders_df.repartition(8, "customer_id")
print(f"After repartition: {orders_repartitioned.rdd.getNumPartitions()} partitions")

# COALESCE → only reduces partitions (no full shuffle, faster)
# Use when: you need FEWER partitions before writing
orders_coalesced = orders_df.coalesce(2)
print(f"After coalesce: {orders_coalesced.rdd.getNumPartitions()} partitions")

print("✅ Repartition vs Coalesce done")

In [ ]:
# ── 4. HANDLING SKEWED DATA ───────────────────────────────────
# Skew = one partition has WAY more data than others
# Fix: use salting technique

from pyspark.sql.functions import concat, lit, rand, floor

# Add a salt key to distribute skewed data evenly
NUM_SALTS = 5

orders_salted = orders_df \
    .withColumn("salt", (floor(rand() * NUM_SALTS)).cast("int")) \
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt")))

customers_salted = customers_df \
    .withColumn("salt", col("customer_id") % NUM_SALTS) \
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt")))

skew_join = orders_salted.join(customers_salted, "salted_key")

print("✅ Skew handling done")

In [ ]:
# ── 5. PARTITION PRUNING ──────────────────────────────────────
# fact_sales is partitioned by order_date
# Querying with order_date filter = only reads relevant partitions
# Much faster than full table scan

pruned = spark.table(f"{catalog}.gold.fact_sales") \
    .filter(col("order_date") >= "2024-01-01")

print(f"Partition pruned row count: {pruned.count()}")
print("✅ Partition pruning done")

In [ ]:
# ── UNPERSIST CACHE (cleanup) ─────────────────────────────────
orders_df.unpersist()
customers_df.unpersist()
print("✅ Cache cleared")